In [1]:
pip install selenium


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install webdriver-manager

  Using cached python_dotenv-1.0.1-py3-none-any.whl.metadata (23 kB)
Using cached python_dotenv-1.0.1-py3-none-any.whl (19 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install bs4


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import random
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import csv

In [7]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Modo headless
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(options=options)

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd

# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/solana-de-ramirez-valsarte-vendimia-seleccionada/w/2035518"

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Solicitud a la página
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

# Inicializar datos
wine_data = {}

try:
    # Nombre y año
    wine_headline = soup.find(class_='wineHeadline-module__vintage--1UHSo')
    if wine_headline:
        name = wine_headline.find('a').text.strip() if wine_headline.find('a') else 'No disponible'
        year = wine_headline.text.strip().split()[-1]  # Última palabra debería ser el año
    else:
        name = 'No disponible'
        year = 'No disponible'

    # País, región, bodega, tipo de vino, uva
    breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
    country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
    region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
    winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
    wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
    grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'

    # Precio
    price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
    price = price_element.text.replace('€', '').replace('\xa0', '').strip() if price_element else 'No disponible'

    # Valoración
    rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
    rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

    # Notas de sabor
    taste_containers = soup.find_all(class_='tasteNote__textContainer--2xPXc')
    taste_notes = []
    for container in taste_containers:
        taste_keywords = container.find(class_='tasteNote__popularKeywords--1gIa2')
        if taste_keywords:
            taste_notes.append(taste_keywords.text.strip())
    taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

    # Maridajes
    food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
    pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]



    # Guardar datos
    wine_data = {
        'Nombre': name,
        'Año': year,
        'País': country,
        'Región': region,
        'Bodega': winery,
        'Tipo de vino': wine_type,
        'Uva': grape,
        'Precio': price,
        'Valoración': rating,
        'Notas de sabor': taste_notes,
        'Maridajes': ', '.join(pairings),
         }

except Exception as e:
    print(f"Error durante la extracción: {e}")

# Guardar en CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=wine_data.keys())
    writer.writeheader()
    writer.writerow(wine_data)

# Convertir a DataFrame y mostrar los primeros registros
df = pd.DataFrame([wine_data])
print(df.head())

          Nombre            Año    País Región             Bodega  \
0  No disponible  No disponible  España  Rioja  Solana de Ramírez   

  Tipo de vino          Uva         Precio Valoración Notas de sabor  \
0   Vino tinto  Tempranillo  No disponible        3.9  No disponible   

                                Maridajes  
0  Ternera, Cordero, Ternera lechal, Aves  


In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time
import random

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar lista para almacenar todos los datos de vino
all_wine_data = []

# Leer las URLs desde un archivo de texto y eliminar duplicados
with open('url.txt', 'r') as file:
    urls = list(set(file.readlines()))  # Eliminamos URLs duplicadas

# Iterar sobre cada URL
for index, url in enumerate(urls, start=1):
    url = url.strip()  # Eliminar espacios en blanco
    wine_data = {}  # Inicializar datos para cada vino

    print(f"Procesando URL {index}/{len(urls)}: {url}")  # Mostrar el progreso

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Nombre y año
        wine_headline = soup.find(class_='wineHeadline__headline--jH7Dl')
        if wine_headline:
            name = wine_headline.find('h1').text.strip() if wine_headline.find('h1') else 'No disponible'
            year = wine_headline.text.strip().split()[-1] if wine_headline.text.strip().split()[-1].isdigit() else 'No disponible'
        else:
            name = 'No disponible'
            year = 'No disponible'

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find_all('a', class_='breadcrumbs__crumb--3eXue')
        country = breadcrumbs[1].text.strip() if len(breadcrumbs) > 1 else 'No disponible'
        region = breadcrumbs[2].text.strip() if len(breadcrumbs) > 2 else 'No disponible'
        winery = breadcrumbs[3].text.strip() if len(breadcrumbs) > 3 else 'No disponible'
        wine_type = breadcrumbs[4].text.strip() if len(breadcrumbs) > 4 else 'No disponible'
        grape = breadcrumbs[5].text.strip() if len(breadcrumbs) > 5 else 'No disponible'

        # Precio
        price_element = soup.find(class_='winePageHeader__current-price--2QNTh')
        price = price_element.text.replace('€', '').strip() if price_element else 'No disponible'

        # Valoración
        rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
        rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'

        # Notas de sabor
        taste_notes = soup.find_all('div', class_='tasteNote__textContainer--2xPXc')
        taste_notes = ', '.join([note.text.strip() for note in taste_notes]) if taste_notes else 'No disponible'

        # Maridajes
        food_pairings = soup.find_all('div', class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]
        
        # Guardar datos
        wine_data = {
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Notas de sabor': taste_notes,
            'Maridajes': ', '.join(pairings),
        }

        all_wine_data.append(wine_data)  # Agregar datos a la lista

    except Exception as e:
        print(f"Error durante la extracción de {url}: {e}")

    # Esperar entre 2 y 5 segundos antes de la siguiente solicitud
    time.sleep(random.uniform(2, 5))

# Guardar todos los datos en un CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir a DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
print(df.head())